In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Load the cleaned dataset from the Delta table
# Try the table name Umar used — adjust if different
df = spark.table("healthcare_facilities_cleaned")

print(f"Loaded {df.count()} facilities")
print(f"Columns: {df.columns}")

Loaded 10000 facilities
Columns: ['name', 'phone_numbers', 'officialPhone', 'email', 'websites', 'officialWebsite', 'yearEstablished', 'facebookLink', 'twitterLink', 'linkedinLink', 'instagramLink', 'address_line1', 'address_line2', 'address_line3', 'address_city', 'address_stateOrRegion', 'address_zipOrPostcode', 'address_country', 'address_countryCode', 'facilityTypeId', 'operatorTypeId', 'affiliationTypeIds', 'description', 'numberDoctors', 'capacity', 'specialties', 'procedure', 'equipment', 'capability', 'recency_of_page_update', 'distinct_social_media_presence_count', 'affiliated_staff_presence', 'custom_logo_presence', 'number_of_facts_about_the_organization', 'post_metrics_most_recent_social_media_post_date', 'post_metrics_post_count', 'engagement_metrics_n_followers', 'engagement_metrics_n_likes', 'engagement_metrics_n_engagements', 'latitude', 'longitude', 'specialties_parsed', 'procedure_parsed', 'equipment_parsed', 'capability_parsed', 'name_cleaned', 'description_cleaned',

In [0]:
# ============================================================================
# TRUST SCORING ENGINE
# ============================================================================
# Start every facility at 100 points
# Deduct points for contradictions and missing data
# HIGH RISK = -25 | MEDIUM RISK = -15 | LOW RISK = -5
# ============================================================================

print("🔍 Building Trust Scoring Engine...\n")

# --- RULE 1: Surgery without Surgeon (-25) ---
# If facility claims surgery but description/capability mentions no surgeon or anesthesiologist
df = df.withColumn("flag_surgery_no_surgeon",
    (col("has_surgery") == True) & 
    (~col("all_text_lower").contains("surgeon")) & 
    (~col("all_text_lower").contains("anesthesiologist")) &
    (~col("all_text_lower").contains("anaesthesiologist"))
)

# --- RULE 2: ICU without Ventilator (-25) ---
# If facility claims ICU but no ventilator/cardiac monitor/defibrillator mentioned
df = df.withColumn("flag_icu_no_equipment",
    (col("has_icu") == True) & 
    (~col("all_text_lower").contains("ventilator")) & 
    (~col("all_text_lower").contains("cardiac monitor")) &
    (~col("all_text_lower").contains("defibrillator"))
)

# --- RULE 3: Emergency without Basics (-25) ---
# Claims emergency but no blood bank, oxygen, or ambulance mentioned
df = df.withColumn("flag_emergency_no_basics",
    (col("has_emergency") == True) & 
    (~col("all_text_lower").contains("blood bank")) & 
    (~col("all_text_lower").contains("oxygen")) &
    (~col("all_text_lower").contains("ambulance"))
)

# --- RULE 4: 24/7 with only part-time staff (-15) ---
df = df.withColumn("flag_24x7_parttime",
    (col("all_text_lower").contains("24/7") | 
     col("all_text_lower").contains("24x7") |
     col("all_text_lower").contains("round the clock")) & 
    (col("all_text_lower").contains("part-time") | 
     col("all_text_lower").contains("parttime") |
     col("all_text_lower").contains("visiting"))
)

# --- RULE 5: Specialty without Specialist (-15) ---
# Lists advanced specialties like oncology/cardiology but no specialist doctor mentioned
df = df.withColumn("flag_specialty_no_specialist",
    (col("all_text_lower").rlike("oncology|cardiology|neurology|nephrology")) & 
    (~col("all_text_lower").contains("specialist")) &
    (~col("all_text_lower").contains("doctor")) &
    (~col("all_text_lower").contains("consultant")) &
    (col("numberDoctors") == 0)
)

# --- RULE 6: Bed Count Mismatch (-5) ---
# Clinic claiming more than 50 beds is suspicious
df = df.withColumn("flag_bed_mismatch",
    (col("facilityType") == "clinic") & 
    (col("capacity") > 50)
)

print("✅ All 6 rules created")

🔍 Building Trust Scoring Engine...

✅ All 6 rules created


In [0]:
# ============================================================================
# CALCULATE TRUST SCORE
# ============================================================================

df = df.withColumn("trust_score",
    lit(100)
    - when(col("flag_surgery_no_surgeon") == True, 25).otherwise(0)
    - when(col("flag_icu_no_equipment") == True, 25).otherwise(0)
    - when(col("flag_emergency_no_basics") == True, 25).otherwise(0)
    - when(col("flag_24x7_parttime") == True, 15).otherwise(0)
    - when(col("flag_specialty_no_specialist") == True, 15).otherwise(0)
    - when(col("flag_bed_mismatch") == True, 5).otherwise(0)
)

# Factor in completeness (low completeness = less trustworthy)
# Deduct up to 10 points for very incomplete records
df = df.withColumn("trust_score",
    col("trust_score") - when(col("completeness_score") < 25, 10)
                          .when(col("completeness_score") < 50, 5)
                          .otherwise(0)
)

# Clamp between 0 and 100
df = df.withColumn("trust_score",
    when(col("trust_score") < 0, 0)
    .when(col("trust_score") > 100, 100)
    .otherwise(col("trust_score"))
)

# Add trust label
df = df.withColumn("trust_label",
    when(col("trust_score") >= 75, "GREEN")
    .when(col("trust_score") >= 40, "YELLOW")
    .otherwise("RED")
)

# ============================================================================
# RESULTS SUMMARY
# ============================================================================
print("=== TRUST SCORE RESULTS ===\n")

print("📊 Score Distribution:")
df.groupBy("trust_label").count().orderBy("trust_label").show()

print("📊 Flag Counts:")
print(f"  Rule 1 - Surgery without Surgeon:    {df.filter(col('flag_surgery_no_surgeon')).count()}")
print(f"  Rule 2 - ICU without Equipment:      {df.filter(col('flag_icu_no_equipment')).count()}")
print(f"  Rule 3 - Emergency without Basics:    {df.filter(col('flag_emergency_no_basics')).count()}")
print(f"  Rule 4 - 24/7 with Part-time:         {df.filter(col('flag_24x7_parttime')).count()}")
print(f"  Rule 5 - Specialty no Specialist:     {df.filter(col('flag_specialty_no_specialist')).count()}")
print(f"  Rule 6 - Bed Count Mismatch:          {df.filter(col('flag_bed_mismatch')).count()}")

print(f"\n📊 Average Trust Score: {df.select(avg('trust_score')).first()[0]:.1f}")
print(f"📊 Min Trust Score: {df.select(min('trust_score')).first()[0]}")
print(f"📊 Max Trust Score: {df.select(max('trust_score')).first()[0]}")

=== TRUST SCORE RESULTS ===

📊 Score Distribution:
+-----------+-----+
|trust_label|count|
+-----------+-----+
|      GREEN| 9320|
|        RED|  133|
|     YELLOW|  547|
+-----------+-----+

📊 Flag Counts:
  Rule 1 - Surgery without Surgeon:    1844
  Rule 2 - ICU without Equipment:      313
  Rule 3 - Emergency without Basics:    680
  Rule 4 - 24/7 with Part-time:         8
  Rule 5 - Specialty no Specialist:     572
  Rule 6 - Bed Count Mismatch:          2

📊 Average Trust Score: 91.6
📊 Min Trust Score: 10
📊 Max Trust Score: 100


In [0]:
# ============================================================================
# SPOT CHECK: Show some RED flagged facilities
# ============================================================================

print("🚨 Sample RED-flagged Facilities:\n")

display(df.filter(col("trust_label") == "RED").select(
    "name_cleaned",
    "city",
    "state",
    "facilityType",
    "trust_score",
    "flag_surgery_no_surgeon",
    "flag_icu_no_equipment",
    "flag_emergency_no_basics",
    "flag_specialty_no_specialist",
    "has_surgery",
    "has_icu",
    "has_emergency"
).limit(10))

🚨 Sample RED-flagged Facilities:



name_cleaned,city,state,facilityType,trust_score,flag_surgery_no_surgeon,flag_icu_no_equipment,flag_emergency_no_basics,flag_specialty_no_specialist,has_surgery,has_icu,has_emergency
Baba Mohan Ram Hospital,Greater Noida,Uttar Pradesh,hospital,25,true,true,true,false,true,true,true
Bharat Hospital,Mehsana,Gujarat,hospital,35,true,false,true,true,true,false,true
"Bhavik ENT Care Clinic, Ghatkopar East, Mumbai",Mumbai,Maharashtra,clinic,25,true,true,true,false,true,true,true
BMH Kozhikode,Kozhikode,Kerala,hospital,25,true,true,true,false,true,true,true
Braham Jyoti Hospital,Hajipur,Bihar,hospital,35,true,false,true,true,true,true,true
CALS - Dr. Chetan's Advanced Laproscopy & Surgical Gastroenterology,Hyderabad,Telangana,clinic,35,true,false,true,true,true,false,true
Care Super Specialty Hospital,Vadodara,Gujarat,hospital,35,true,false,true,true,true,true,true
"Cataract and Cornea Clinic, Meera Nagar, Udaipur",Udaipur,Rajasthan,clinic,25,true,true,true,false,true,true,true
Cawnpore Med Hospital,Kanpur Nagar,Uttar Pradesh,hospital,10,true,true,true,true,true,true,true
Centre for Neuro Spinal Injury,Mathura,Uttar Pradesh,clinic,35,true,false,true,true,true,false,true


In [0]:
# ============================================================================
# SAVE TRUST-SCORED DATASET
# ============================================================================

# Overwrite the cleaned table with trust scores added
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("healthcare_facilities_cleaned")

print(f"✅ Saved {df.count()} facilities with trust scores")
print(f"📊 New columns added: trust_score, trust_label, 6 flag columns")

✅ Saved 10000 facilities with trust scores
📊 New columns added: trust_score, trust_label, 6 flag columns
